In [1]:
# 环境准备
import duckdb
import pandas as pd
import numpy as np

# 创建 DuckDB 连接（内存数据库）
conn = duckdb.connect(':memory:')

print("DuckDB 环境准备完成！")

DuckDB 环境准备完成！


In [2]:
# DuckDB 支持直接执行多语句
# 如果表已存在则删除（避免重复执行报错）
conn.execute("DROP TABLE IF EXISTS employee_attendance")
conn.execute("""
    -- 创建员工考勤表
    CREATE TABLE employee_attendance (
        emp_id INTEGER,
        emp_name VARCHAR(50),
        department VARCHAR(30),
        hire_date DATE,
        clock_in TIMESTAMP,
        clock_out TIMESTAMP
    );

    -- 插入数据
    INSERT INTO employee_attendance VALUES
    (101, 'Alice',   'Engineering', '2024-03-15', '2026-06-23 08:55:00', '2026-06-23 17:30:00'),
    (101, 'Alice',   'Engineering', '2024-03-15', '2026-06-24 09:10:00', '2026-06-24 18:05:00'),
    (101, 'Alice',   'Engineering', '2024-03-15', '2026-06-25 08:45:00', '2026-06-25 17:00:00'),
    (102, 'Bob',     'Marketing',   '2025-01-10', '2026-06-23 09:30:00', '2026-06-23 18:15:00'),
    (102, 'Bob',     'Marketing',   '2025-01-10', '2026-06-24 09:00:00', '2026-06-24 17:45:00'),
    (102, 'Bob',     'Marketing',   '2025-01-10', '2026-06-25 08:50:00', NULL),
    (103, 'Charlie', 'Engineering', '2026-01-20', '2026-06-23 10:00:00', '2026-06-23 19:00:00'),
    (103, 'Charlie', 'Engineering', '2026-01-20', '2026-06-24 09:15:00', '2026-06-24 18:30:00'),
    (103, 'Charlie', 'Engineering', '2026-01-20', '2026-06-25 08:30:00', NULL),
    (104, 'Diana',   'HR',          '2025-08-01', '2026-06-23 09:00:00', '2026-06-23 17:00:00'),
    (104, 'Diana',   'HR',          '2025-08-01', '2026-06-24 08:45:00', '2026-06-24 17:15:00'),
    (104, 'Diana',   'HR',          '2025-08-01', '2026-06-25 09:20:00', '2026-06-25 18:00:00');
""")
# 调整显示设置
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 25)
# 验证数据
df = conn.execute("SELECT * FROM employee_attendance").fetchdf()
print("✅ 数据表创建成功！")
print(f"共 {len(df)} 条记录\n")
print(df)

✅ 数据表创建成功！
共 12 条记录

    emp_id emp_name   department  hire_date            clock_in           clock_out
0      101    Alice  Engineering 2024-03-15 2026-06-23 08:55:00 2026-06-23 17:30:00
1      101    Alice  Engineering 2024-03-15 2026-06-24 09:10:00 2026-06-24 18:05:00
2      101    Alice  Engineering 2024-03-15 2026-06-25 08:45:00 2026-06-25 17:00:00
3      102      Bob    Marketing 2025-01-10 2026-06-23 09:30:00 2026-06-23 18:15:00
4      102      Bob    Marketing 2025-01-10 2026-06-24 09:00:00 2026-06-24 17:45:00
5      102      Bob    Marketing 2025-01-10 2026-06-25 08:50:00                 NaT
6      103  Charlie  Engineering 2026-01-20 2026-06-23 10:00:00 2026-06-23 19:00:00
7      103  Charlie  Engineering 2026-01-20 2026-06-24 09:15:00 2026-06-24 18:30:00
8      103  Charlie  Engineering 2026-01-20 2026-06-25 08:30:00                 NaT
9      104    Diana           HR 2025-08-01 2026-06-23 09:00:00 2026-06-23 17:00:00
10     104    Diana           HR 2025-08-01 2026-06-24 

# 📊 员工考勤数据分析练习

> **练习目标**：掌握日期/时间数据的算术运算  
> **技术栈**：DuckDB (SQL) + Pandas (Python)  
> **数据表**：`employee_attendance`

---

## 📋 数据字典

| 列名 | 数据类型 | 说明 |
|------|---------|------|
| `emp_id` | INTEGER | 员工编号 |
| `emp_name` | VARCHAR | 员工姓名 |
| `department` | VARCHAR | 所属部门 |
| `hire_date` | DATE | 入职日期 |
| `clock_in` | TIMESTAMP | 上班打卡时间 |
| `clock_out` | TIMESTAMP | 下班打卡时间（NULL 表示未打卡） |

---


## 🎯 练习任务1

### 查询 ① 上班时长计算

**需求**：计算每位员工每次的实际上班时长

| 输出列 | 说明 | 数据类型 |
|--------|------|---------|
| `emp_name` | 员工姓名 | VARCHAR |
| `work_date` | 上班日期（只取日期部分） | DATE |
| `work_duration` | 上班时长 | INTERVAL |
| `work_minutes` | 上班分钟数 | NUMERIC |

**要点提示**：
- `timestamp - timestamp` → 返回 INTERVAL
- 用 `EXTRACT(EPOCH FROM interval) / 60` 将 INTERVAL 转为分钟数
- 排除 `clock_out` 为 NULL 的记录

---

In [24]:
# SQL轨道
query1 = """
SELECT  emp_name,
        clock_in::DATE AS work_date,
        (clock_out - clock_in) AS work_duration,
        EXTRACT(EPOCH FROM (clock_out - clock_in)) / 60 AS work_minutes
FROM employee_attendance
WHERE clock_out IS NOT NULL
ORDER BY emp_name,work_date
"""
result1 = conn.execute(query1).fetchdf()
print("📊 查询结果：每位员工每次上班时长")
result1

📊 查询结果：每位员工每次上班时长


,emp_name,work_date,work_duration,work_minutes
0,Alice,2026-06-23,0 days 08:35:00,515.0
1,Alice,2026-06-24,0 days 08:55:00,535.0
2,Alice,2026-06-25,0 days 08:15:00,495.0
3,Bob,2026-06-23,0 days 08:45:00,525.0
4,Bob,2026-06-24,0 days 08:45:00,525.0
5,Charlie,2026-06-23,0 days 09:00:00,540.0
6,Charlie,2026-06-24,0 days 09:15:00,555.0
7,Diana,2026-06-23,0 days 08:00:00,480.0
8,Diana,2026-06-24,0 days 08:30:00,510.0
9,Diana,2026-06-25,0 days 08:40:00,520.0


In [28]:
# PANDAS轨道
df_filtered = df.loc[df['clock_out'].notna(),['emp_name','clock_in','clock_out']].copy()
df_work = (
    df_filtered
    .assign(
        work_date = lambda x:x['clock_in'].dt.floor('D'),# 因为SQL通过conn.execute(query1)转换了一次，日期被带上了时间精度，所以用.floor('D'),让这里的日期也带上时间精度，否则两表对账会报错
        work_duration = lambda x:x['clock_out'] - x['clock_in'],
        work_minutes = lambda x:((x['clock_out'] - x['clock_in']).dt.total_seconds()/60).round(2)
    )
    .loc[:,['emp_name','work_date','work_duration','work_minutes']]
    .sort_values(by=['emp_name','work_date'])
    .reset_index(drop=True)
)

print(df_work)

  emp_name  work_date   work_duration  work_minutes
0    Alice 2026-06-23 0 days 08:35:00         515.0
1    Alice 2026-06-24 0 days 08:55:00         535.0
2    Alice 2026-06-25 0 days 08:15:00         495.0
3      Bob 2026-06-23 0 days 08:45:00         525.0
4      Bob 2026-06-24 0 days 08:45:00         525.0
5  Charlie 2026-06-23 0 days 09:00:00         540.0
6  Charlie 2026-06-24 0 days 09:15:00         555.0
7    Diana 2026-06-23 0 days 08:00:00         480.0
8    Diana 2026-06-24 0 days 08:30:00         510.0
9    Diana 2026-06-25 0 days 08:40:00         520.0


In [29]:
# 两表对账
pd.testing.assert_frame_equal(
    result1.reset_index(drop=True),
    df_work.reset_index(drop=True),
    check_dtype=False
)
print('✅完美对账！！！')

✅完美对账！！！
